In [11]:
import tensorflow  as tf
from tensorflow import keras
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import numpy as np
import os
import matplotlib.pyplot as plt
from PIL import Image

In [1]:
!pip install -q kaggle
import os, json

os.environ['KAGGLE_USERNAME'] = "tajwerfatima"
os.environ['KAGGLE_KEY'] = "KGAT_a8a1c3955b19b6f4831d741db0efbf9f"

!mkdir -p ~/.kaggle
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": os.environ['KAGGLE_USERNAME'],
               "key": os.environ['KAGGLE_KEY']}, f)
!chmod 600 /root/.kaggle/kaggle.json

print("✅ Kaggle ready!")

✅ Kaggle ready!


In [2]:
!kaggle datasets list -s "skin disease classification"

ref                                                                title                                                      size  lastUpdated                 downloadCount  voteCount  usabilityRating  
-----------------------------------------------------------------  --------------------------------------------------  -----------  --------------------------  -------------  ---------  ---------------  
trainingdatapro/skin-defects-acne-redness-and-bags-under-the-eyes  Skin Disease Classification Dataset                   267926468  2023-11-16 17:13:43.830000           4534         54  1.0              
pritpal2873/multiple-skin-disease-detection-and-classification     Multiple Skin Disease Detection and Classification    822659080  2024-06-28 04:19:05.553000           2493         26  0.875            
riyaelizashaju/skin-disease-classification-image-dataset           Skin Disease Classification [Image Dataset]           177107921  2023-03-14 05:09:17.910000           6565         31

In [6]:
!kaggle datasets download -d subirbiswas19/skin-disease-dataset
!unzip -q skin-disease-dataset.zip -d /content/skin_data2

import os
for root, dirs, files in os.walk('/content/skin_data2'):
    level = root.replace('/content/skin_data2', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}📁 {os.path.basename(root)}/')
        if files:
            print(f'{indent}  → {len(files)} images')

Dataset URL: https://www.kaggle.com/datasets/subirbiswas19/skin-disease-dataset
License(s): CC0-1.0
  0% 0.00/17.3M [00:00<?, ?B/s]
100% 17.3M/17.3M [00:00<00:00, 1.17GB/s]
📁 skin_data2/
  📁 skin-disease-datasaet/
    📁 test_set/
    📁 train_set/


In [7]:
# Cell 3: Explore
import os

for root, dirs, files in os.walk('/content/skin_data2'):
    level = root.replace('/content/skin_data', '').count(os.sep)
    if level < 3:
        indent = ' ' * 2 * level
        print(f'{indent}📁 {os.path.basename(root)}/')
        if files:
            print(f'{indent}  → {len(files)} images')

📁 skin_data2/
  📁 skin-disease-datasaet/
    📁 test_set/
    📁 train_set/


In [9]:
import os

base = '/content/skin_data2/skin-disease-datasaet'

for split in ['train_set', 'test_set']:
    split_path = os.path.join(base, split)
    print(f"\n📁 {split}:")

    categories = os.listdir(split_path)
    for cat in categories:
        cat_path = os.path.join(split_path, cat)
        if os.path.isdir(cat_path):
            count = len(os.listdir(cat_path))
            print(f"   {cat}: {count} images")


📁 train_set:
   PA-cutaneous-larva-migrans: 100 images
   FU-athlete-foot: 124 images
   FU-nail-fungus: 129 images
   BA- cellulitis: 136 images
   VI-shingles: 130 images
   VI-chickenpox: 136 images
   BA-impetigo: 80 images
   FU-ringworm: 90 images

📁 test_set:
   PA-cutaneous-larva-migrans: 25 images
   FU-athlete-foot: 32 images
   FU-nail-fungus: 33 images
   BA- cellulitis: 34 images
   VI-shingles: 33 images
   VI-chickenpox: 34 images
   BA-impetigo: 20 images
   FU-ringworm: 23 images


In [12]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

base = '/content/skin_data2/skin-disease-datasaet'

# Train generator - WITH augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True
)

# Test generator - ONLY rescale
test_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    f'{base}/train_set',  # ← its own folder!
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    f'{base}/test_set',   # ← its own folder!
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical',
    shuffle=False  # ← remember Day 9!
)

print(f"✅ Categories: {list(train_generator.class_indices.keys())}")

Found 924 images belonging to 8 classes.
Found 233 images belonging to 8 classes.
✅ Categories: ['BA- cellulitis', 'BA-impetigo', 'FU-athlete-foot', 'FU-nail-fungus', 'FU-ringworm', 'PA-cutaneous-larva-migrans', 'VI-chickenpox', 'VI-shingles']


In [23]:
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras import layers, models

# Load pre-trained MobileNetV2
base_model = MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

# Freeze base model
base_model.trainable = False

# Build classifier
model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dropout(0.3),
    layers.Dense(8, activation='softmax')  # 48classes!
])

# Compile
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("✅ Model built!")
model.summary()

✅ Model built!


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ mobilenetv2_1.00_224            │ (None, 7, 7, 1280)     │     2,257,984 │
│ (Functional)                    │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d_1      │ (None, 1280)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 1280)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │        10,248 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,268,232 (8.65 MB)

 Trainable params: 10,248 (40.03 KB)

 Non-trainable params: 2,257,984 (8.61 MB)

In [25]:
# Cell 6: Train!
print("🚗 Training vehicles classifier...\n")

history = model.fit(
    train_generator,
    epochs=10,
    validation_data = test_generator
)

print("\n✅ Training complete!")

🚗 Training vehicles classifier...

Epoch 1/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.8905 - loss: 0.3197 - val_accuracy: 0.9227 - val_loss: 0.2894
Epoch 2/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 76s 3s/step - accuracy: 0.9097 - loss: 0.2964 - val_accuracy: 0.9056 - val_loss: 0.2870
Epoch 3/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 65s 2s/step - accuracy: 0.8971 - loss: 0.3060 - val_accuracy: 0.9056 - val_loss: 0.2796
Epoch 4/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.9167 - loss: 0.2766 - val_accuracy: 0.9099 - val_loss: 0.2587
Epoch 5/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.8999 - loss: 0.2924 - val_accuracy: 0.9270 - val_loss: 0.2599
Epoch 6/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.9252 - loss: 0.2448 - val_accuracy: 0.9270 - val_loss: 0.2606
Epoch 7/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 96s 3s/step - accuracy: 0.9400 - loss: 0.2032 - val_accuracy: 0.9013 - val_loss: 0.2753
Epoch 8/10
29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.9256 - loss: 0.2502

In [ ]:
# 🚗 Training vehicles classifier...

# Epoch 1/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 84s 3s/step - accuracy: 0.2658 - loss: 2.1829 - val_accuracy: 0.6781 - val_loss: 1.0250
# Epoch 2/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.6222 - loss: 1.0908 - val_accuracy: 0.7768 - val_loss: 0.6824
# Epoch 3/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 66s 2s/step - accuracy: 0.7727 - loss: 0.7221 - val_accuracy: 0.8498 - val_loss: 0.5028
# Epoch 4/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 62s 2s/step - accuracy: 0.8240 - loss: 0.5477 - val_accuracy: 0.8498 - val_loss: 0.4681
# Epoch 5/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 84s 2s/step - accuracy: 0.8513 - loss: 0.4737 - val_accuracy: 0.8798 - val_loss: 0.4183
# Epoch 6/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.8480 - loss: 0.4610 - val_accuracy: 0.8927 - val_loss: 0.3728
# Epoch 7/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.8659 - loss: 0.4462 - val_accuracy: 0.8670 - val_loss: 0.3582
# Epoch 8/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 64s 2s/step - accuracy: 0.8664 - loss: 0.4333 - val_accuracy: 0.8798 - val_loss: 0.3574
# Epoch 9/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 63s 2s/step - accuracy: 0.9053 - loss: 0.3545 - val_accuracy: 0.8841 - val_loss: 0.3423
# Epoch 10/10
# 29/29 ━━━━━━━━━━━━━━━━━━━━ 95s 3s/step - accuracy: 0.8941 - loss: 0.3542 - val_accuracy: 0.8798 - val_loss: 0.3371

# ✅ Training complete!


In [ ]:
# Cell 5: Visualize Sample Images
import matplotlib.pyplot as plt
from PIL import Image
import random

categories = ['sunflower', 'daisy' ,'tulip']
clean_dir = '/content/clean_flowers'

fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for i, category in enumerate(categories):
    category_path = os.path.join(clean_dir, category)
    image_files = os.listdir(category_path)

    for j in range(4):
        ax = axes[i, j]
        random_image = random.choice(image_files)
        img_path = os.path.join(category_path, random_image)

        img = Image.open(img_path)
        ax.imshow(img)
        ax.axis('off')
        if j == 0:
            ax.set_title(category.upper(), fontsize=14, fontweight='bold')

plt.suptitle('Flower dataset - sunflower, daisy , rose , tulip ' ,fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

print("✅ Your balanced Flower dataset!")
print("Sunflower 🌻")
print("Daisy 🌼")
print("Tulip 🌷")

In [ ]:
done !! its a good education point ill kepp it in mind !! thanks ....also we had like 8 output shape and 8 input shape that what i had to change mostly..other than that it was same ....🚗 Training vehicles classifier... 🚗 Training vehicles classifier.....first time i trained it started from 20 percent accuracy went to 90 percent in epoch 9 and went to 89 percent in final epoch also val accuracy of  87 percent  in final epoch ..i was not satisfied so i trained it again ...second time it started with 85 percent accyracy and 92 percent validation accuracy and ended on 93 percent training and 92 percent validation accuracy (Same??)

In [28]:
# Cell 7: FIXED Per-Class Metrics
from sklearn.metrics import classification_report
import numpy as np

# RESET THE GENERATOR FIRST!!! (This was missing!)
test_generator.reset()

# Now predict
val_predictions = model.predict(test_generator)
val_pred_classes = np.argmax(val_predictions, axis=1)

# Get true labels AFTER reset
val_true_classes = test_generator.classes

category_names = list(test_generator.class_indices.keys())

print("="*70)
print("PER-CLASS PERFORMANCE ANALYSIS")
print("="*70)
print(classification_report(val_true_classes, val_pred_classes,
                           target_names=category_names))

8/8 ━━━━━━━━━━━━━━━━━━━━ 15s 2s/step
PER-CLASS PERFORMANCE ANALYSIS
                            precision    recall  f1-score   support

            BA- cellulitis       0.92      1.00      0.96        33
               BA-impetigo       0.95      1.00      0.98        20
           FU-athlete-foot       0.86      1.00      0.93        32
            FU-nail-fungus       0.97      0.94      0.95        33
               FU-ringworm       0.86      0.83      0.84        23
PA-cutaneous-larva-migrans       0.87      0.80      0.83        25
             VI-chickenpox       1.00      1.00      1.00        34
               VI-shingles       0.93      0.79      0.85        33

                  accuracy                           0.92       233
                 macro avg       0.92      0.92      0.92       233
              weighted avg       0.92      0.92      0.92       233

